In [2]:
# =============================================================================
# MCX SPAN DUMMY MODEL
# STEP 1: DATA LAYER
# Loads portfolio from Excel, builds Instrument + CombinedCommodity objects
# =============================================================================

import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# =============================================================================
# 1.1 SPAN PARAMETERS
# MCX-style parameters for each commodity.
# Price Scan Range (PSR) = max % price move SPAN assumes for 1 day
# Vol Scan Range (VSR)   = absolute vol shift for option repricing
# SOM Rate               = Short Option Minimum as % of contract value
# Spread Charge          = fixed ₹ charge per calendar spread pair
# =============================================================================

SPAN_PARAMS: Dict[str, dict] = {
    "Turmeric": {
        "price_scan_pct" : 0.06,      # 6% max daily move
        "vol_scan_up"    : 0.04,      # +4 vol points
        "vol_scan_down"  : 0.04,      # -4 vol points
        "som_rate"       : 0.015,     # 1.5% of contract value
        "spread_charge"  : 5000,      # ₹5,000 per calendar spread pair
        "lot_size_qt"    : 50,        # 5 MT × 10 qt/MT = 50 quintals per lot
    },
    "Coriander": {
        "price_scan_pct" : 0.06,
        "vol_scan_up"    : 0.04,
        "vol_scan_down"  : 0.04,
        "som_rate"       : 0.015,
        "spread_charge"  : 5000,
        "lot_size_qt"    : 50,
    },
    "Jeera": {
        "price_scan_pct" : 0.06,
        "vol_scan_up"    : 0.04,
        "vol_scan_down"  : 0.04,
        "som_rate"       : 0.015,
        "spread_charge"  : 5000,
        "lot_size_qt"    : 30,        # 3 MT × 10 = 30 quintals per lot
    },
    "Guar Gum": {
        "price_scan_pct" : 0.06,
        "vol_scan_up"    : 0.04,
        "vol_scan_down"  : 0.04,
        "som_rate"       : 0.015,
        "spread_charge"  : 4000,
        "lot_size_qt"    : 50,
    },
    "Guar Seed": {
        "price_scan_pct" : 0.06,
        "vol_scan_up"    : 0.04,
        "vol_scan_down"  : 0.04,
        "som_rate"       : 0.015,
        "spread_charge"  : 4000,
        "lot_size_qt"    : 50,
    },
    "Castor Seed": {
        "price_scan_pct" : 0.06,
        "vol_scan_up"    : 0.04,
        "vol_scan_down"  : 0.04,
        "som_rate"       : 0.015,
        "spread_charge"  : 4000,
        "lot_size_qt"    : 50,
    },
    "Cotton Seed Oil Cake": {
        "price_scan_pct" : 0.05,
        "vol_scan_up"    : 0.03,
        "vol_scan_down"  : 0.03,
        "som_rate"       : 0.015,
        "spread_charge"  : 3000,
        "lot_size_qt"    : 100,       # 10 MT × 10 = 100 quintals per lot
    },
    "Kapas (Cotton)": {
        "price_scan_pct" : 0.05,
        "vol_scan_up"    : 0.03,
        "vol_scan_down"  : 0.03,
        "som_rate"       : 0.015,
        "spread_charge"  : 3000,
        "lot_size_qt"    : 40,        # 4 MT × 10 = 40 quintals per lot
    },
}

# Inter-CC spread credit rates
# These pairs get a partial margin credit because they are correlated
INTER_CC_SPREADS: List[dict] = [
    {
        "cc_group"    : "Guar CC",
        "leg1"        : "Guar Gum",
        "leg2"        : "Guar Seed",
        "credit_rate" : 0.30,     # 30% credit on smaller net delta side
    },
    {
        "cc_group"    : "Cotton CC",
        "leg1"        : "Cotton Seed Oil Cake",
        "leg2"        : "Kapas (Cotton)",
        "credit_rate" : 0.30,
    },
]

# =============================================================================
# 1.2 INSTRUMENT OBJECT
# One object per row in the Excel file
# =============================================================================

@dataclass
class Instrument:
    row_id:          int
    commodity:       str
    cc_group:        str
    inst_type:       str           # 'Future' or 'Option'
    direction:       str           # 'Buy' or 'Sell'
    qty_lots:        int
    contract_month:  str
    lot_size_qt:     float         # quintals per lot (derived from MT)

    # Futures fields
    futures_price:   Optional[float] = None

    # Options fields
    option_premium:  Optional[float] = None
    strike_price:    Optional[float] = None
    option_type:     Optional[str]   = None   # 'Call' or 'Put'
    implied_vol:     Optional[float] = None   # as decimal e.g. 0.28
    delta:           Optional[float] = None

    # SPAN tags
    span_role:       str = ""
    span_component:  str = ""

    # Derived
    signed_qty:      int = 0       # +ve = long, -ve = short

    def __post_init__(self):
        self.signed_qty = (
            self.qty_lots if self.direction == "Buy" else -self.qty_lots
        )

    @property
    def contract_value(self) -> float:
        """
        MCX contract value in ₹.
        = futures_price (₹/qt) × lot_size_qt × qty_lots
        """
        if self.inst_type == "Future" and self.futures_price:
            return self.futures_price * self.lot_size_qt * self.qty_lots
        elif self.inst_type == "Option" and self.option_premium:
            return self.option_premium * self.lot_size_qt * self.qty_lots
        return 0.0

    @property
    def is_short(self) -> bool:
        return self.direction == "Sell"

    @property
    def is_long(self) -> bool:
        return self.direction == "Buy"


# =============================================================================
# 1.3 COMBINED COMMODITY OBJECT
# Groups all instruments sharing a CC group
# =============================================================================

@dataclass
class CombinedCommodity:
    name:        str                          # e.g. "Spices CC"
    instruments: List[Instrument] = field(default_factory=list)

    def add(self, inst: Instrument):
        self.instruments.append(inst)

    @property
    def futures(self) -> List[Instrument]:
        return [i for i in self.instruments if i.inst_type == "Future"]

    @property
    def options(self) -> List[Instrument]:
        return [i for i in self.instruments if i.inst_type == "Option"]

    @property
    def short_options(self) -> List[Instrument]:
        return [i for i in self.options if i.is_short]

    @property
    def long_options(self) -> List[Instrument]:
        return [i for i in self.options if i.is_long]

    @property
    def net_delta(self) -> float:
        """
        Net dollar delta of the CC group.
        Future delta = signed_qty × lot_size_qt × futures_price × 0.95
        Option delta = signed_qty × lot_size_qt × option_premium × |delta|
        """
        total = 0.0
        for inst in self.instruments:
            if inst.inst_type == "Future":
                total += (
                    inst.signed_qty
                    * inst.lot_size_qt
                    * inst.futures_price
                    * 0.95          # futures delta ≈ 0.95 per portfolio
                )
            elif inst.inst_type == "Option":
                total += (
                    inst.signed_qty
                    * inst.lot_size_qt
                    * abs(inst.delta or 0)
                    * (inst.futures_price or inst.strike_price or 0)
                )
        return round(total, 2)

    @property
    def commodities(self) -> List[str]:
        """Unique commodity names within this CC group."""
        return list(dict.fromkeys(i.commodity for i in self.instruments))


# =============================================================================
# 1.4 PORTFOLIO OBJECT
# Top-level container — holds all CombinedCommodities
# =============================================================================

@dataclass
class Portfolio:
    combined_commodities: Dict[str, CombinedCommodity] = field(
        default_factory=dict
    )

    def add_instrument(self, inst: Instrument):
        if inst.cc_group not in self.combined_commodities:
            self.combined_commodities[inst.cc_group] = CombinedCommodity(
                name=inst.cc_group
            )
        self.combined_commodities[inst.cc_group].add(inst)

    @property
    def all_instruments(self) -> List[Instrument]:
        result = []
        for cc in self.combined_commodities.values():
            result.extend(cc.instruments)
        return result

    @property
    def total_contract_value(self) -> float:
        return sum(i.contract_value for i in self.all_instruments)


# =============================================================================
# 1.5 PORTFOLIO LOADER
# Reads the Excel file and builds the full object hierarchy
# =============================================================================

def load_portfolio(path: str) -> Portfolio:
    """
    Reads SPAN_Portfolio sheet from the Excel file.
    Returns a fully populated Portfolio object.
    """
    df = pd.read_excel(path, sheet_name="SPAN_Portfolio")
    df.columns = [
        "Row", "Commodity", "CC_Group", "Inst_Type", "Buy_Sell",
        "Qty_Lots", "Contract_Month", "Futures_Price", "Option_Premium",
        "Strike_Price", "Option_Type", "Implied_Vol_Pct", "Lot_Size_MT",
        "Contract_Value", "Delta", "SPAN_Role", "SPAN_Component"
    ]

    portfolio = Portfolio()

    for _, row in df.iterrows():
        commodity = str(row["Commodity"]).strip()
        params    = SPAN_PARAMS.get(commodity, {})

        # lot size in quintals: MT × 10 qt/MT
        lot_size_mt = float(row["Lot_Size_MT"])
        lot_size_qt = lot_size_mt * 10.0

        inst = Instrument(
            row_id         = int(row["Row"]),
            commodity      = commodity,
            cc_group       = str(row["CC_Group"]).strip(),
            inst_type      = str(row["Inst_Type"]).strip(),
            direction      = str(row["Buy_Sell"]).strip(),
            qty_lots       = int(row["Qty_Lots"]),
            contract_month = str(row["Contract_Month"]).strip(),
            lot_size_qt    = lot_size_qt,
            futures_price  = float(row["Futures_Price"])
                             if pd.notna(row["Futures_Price"]) else None,
            option_premium = float(row["Option_Premium"])
                             if pd.notna(row["Option_Premium"]) else None,
            strike_price   = float(row["Strike_Price"])
                             if pd.notna(row["Strike_Price"]) else None,
            option_type    = str(row["Option_Type"]).strip()
                             if pd.notna(row["Option_Type"]) else None,
            implied_vol    = float(row["Implied_Vol_Pct"]) / 100.0
                             if pd.notna(row["Implied_Vol_Pct"]) else None,
            delta          = float(row["Delta"])
                             if pd.notna(row["Delta"]) else None,
            span_role      = str(row["SPAN_Role"]).strip()
                             if pd.notna(row["SPAN_Role"]) else "",
            span_component = str(row["SPAN_Component"]).strip()
                             if pd.notna(row["SPAN_Component"]) else "",
        )
        portfolio.add_instrument(inst)

    return portfolio


# =============================================================================
# 1.6 VALIDATION PRINTER
# =============================================================================

def print_portfolio_summary(portfolio: Portfolio):
    print("=" * 70)
    print("MCX SPAN DUMMY MODEL — PORTFOLIO LOADED")
    print("=" * 70)

    all_inst = portfolio.all_instruments
    futures  = [i for i in all_inst if i.inst_type == "Future"]
    options  = [i for i in all_inst if i.inst_type == "Option"]

    print(f"\n  Total instruments : {len(all_inst)}")
    print(f"  Futures           : {len(futures)}")
    print(f"  Options           : {len(options)}")
    print(f"  CC Groups         : {len(portfolio.combined_commodities)}")

    for cc_name, cc in portfolio.combined_commodities.items():
        print(f"\n  {'─'*60}")
        print(f"  CC GROUP : {cc_name}")
        print(f"  {'─'*60}")
        print(f"  {'Row':<4} {'Commodity':<24} {'Type':<7} {'Dir':<5} "
              f"{'Qty':>4} {'Month':<8} {'Price/Prem':>12} "
              f"{'LotQt':>6} {'ContractVal':>14}")
        print(f"  {'-'*88}")

        for inst in cc.instruments:
            price_str = (
                f"₹{inst.futures_price:>8,.0f}"
                if inst.futures_price
                else f"₹{inst.option_premium:>8,.0f} (prem)"
            )
            print(f"  {inst.row_id:<4} {inst.commodity:<24} "
                  f"{inst.inst_type:<7} {inst.direction:<5} "
                  f"{inst.qty_lots:>4} {inst.contract_month:<8} "
                  f"{price_str:>12} {inst.lot_size_qt:>6.0f} "
                  f"₹{inst.contract_value:>13,.0f}")

        print(f"\n  Futures : {len(cc.futures)}  "
              f"Options : {len(cc.options)}  "
              f"(Long opt: {len(cc.long_options)}, "
              f"Short opt: {len(cc.short_options)})")

    print(f"\n  {'='*60}")
    print(f"  CONTRACT VALUE RECONCILIATION")
    print(f"  {'─'*60}")
    print(f"  {'Commodity':<24} {'Qty':>4} {'Price(₹/qt)':>12} "
          f"{'LotQts':>7} {'Contract Val':>14}")
    print(f"  {'-'*65}")
    for inst in futures:
        print(f"  {inst.commodity:<24} {inst.qty_lots:>4} "
              f"₹{inst.futures_price:>10,.0f} {inst.lot_size_qt:>7.0f} "
              f"₹{inst.contract_value:>13,.0f}")

    print(f"\n  SHORT OPTIONS (SOM candidates):")
    print(f"  {'Row':<4} {'Commodity':<24} {'Type':<5} {'Strike':>8} "
          f"{'Prem':>6} {'Delta':>7} {'Role'}")
    print(f"  {'-'*80}")
    for inst in options:
        if inst.is_short:
            som_flag = " ★ SOM TRIGGER" if "SOM TRIGGER" in inst.span_role else ""
            print(f"  {inst.row_id:<4} {inst.commodity:<24} "
                  f"{inst.option_type:<5} ₹{inst.strike_price:>7,.0f} "
                  f"₹{inst.option_premium:>5.0f} {inst.delta:>7.2f}"
                  f"{som_flag}")

    print(f"\n  CALENDAR SPREAD PAIRS DETECTED:")
    for cc_name, cc in portfolio.combined_commodities.items():
        by_comm: Dict[str, List[Instrument]] = {}
        for inst in cc.futures:
            if inst.commodity not in by_comm:
                by_comm[inst.commodity] = []
            by_comm[inst.commodity].append(inst)
        for comm, insts in by_comm.items():
            months = set(i.contract_month for i in insts)
            if len(months) > 1:
                print(f"  ✅ {comm} ({cc_name}): "
                      f"{', '.join(f'{i.direction} {i.contract_month} ×{i.qty_lots}' for i in insts)}")

    print(f"\n{'='*70}")
    print(f"  Step 1 complete — Portfolio object hierarchy ready.")
    print(f"  Next: Step 2 — 16 SPAN Scenarios")
    print(f"{'='*70}")


# =============================================================================
# 1.7 RUN
# =============================================================================

PORTFOLIO_PATH = "/content/FUTCOM_SPAN_Enhanced_Portfolio.xlsx"

if __name__ == "__main__":
    portfolio = load_portfolio(PORTFOLIO_PATH)
    print_portfolio_summary(portfolio)

MCX SPAN DUMMY MODEL — PORTFOLIO LOADED

  Total instruments : 22
  Futures           : 12
  Options           : 10
  CC Groups         : 4

  ────────────────────────────────────────────────────────────
  CC GROUP : Spices CC
  ────────────────────────────────────────────────────────────
  Row  Commodity                Type    Dir    Qty Month      Price/Prem  LotQt    ContractVal
  ----------------------------------------------------------------------------------------
  1    Turmeric                 Future  Buy      1 Apr-26      ₹  16,220     50 ₹      811,000
  2    Coriander                Future  Buy      2 Apr-26      ₹  13,350     50 ₹    1,335,000
  3    Jeera                    Future  Buy      2 Apr-26      ₹  22,285     30 ₹    1,337,100
  9    Turmeric                 Future  Buy      1 Jun-26      ₹  16,450     50 ₹      822,500
  10   Turmeric                 Future  Sell     1 Apr-26      ₹  16,220     50 ₹      811,000
  13   Jeera                    Option  Buy      

In [3]:
# =============================================================================
# MCX SPAN DUMMY MODEL
# STEP 2: 16 SPAN SCENARIOS
# Hard-codes the CME/MCX-style 16 scenario grid
# =============================================================================

# Paste Step 1 code above this before running

# =============================================================================
# 2.1 WHAT ARE THE 16 SCENARIOS?
#
# SPAN tests your portfolio against 16 "what-if" market situations.
# Each scenario combines:
#   - A PRICE MOVE  : fraction of the Price Scan Range (PSR)
#   - A VOL MOVE    : implied volatility shifts UP or DOWN
#
# The 16 scenarios cover:
#   Scenarios 1-14  : Systematic price moves from -PSR to +PSR
#                     at 1/3, 2/3, and full PSR in both directions
#                     crossed with vol up and vol down
#   Scenario 15-16  : EXTREME moves = 2× PSR, but weighted at only 35%
#                     (because they are possible but less likely)
#
# For FUTURES: vol move has NO effect (futures P&L is purely price-driven)
# For OPTIONS: both price AND vol move affect P&L (delta + vega effects)
#
# PSR = Price Scan Range = price × price_scan_pct
# e.g. Turmeric: ₹16,220 × 6% = ₹973.2 max move
# =============================================================================

from dataclasses import dataclass
from typing import List

@dataclass
class Scenario:
    scenario_id:   int
    label:         str
    price_frac:    float   # fraction of PSR: -1.0 to +1.0 (or ±2.0 for extreme)
    vol_move:      str     # "UP" or "DOWN"
    weight:        float   # 1.0 for normal, 0.35 for extreme scenarios

# =============================================================================
# 2.2 THE 16 SCENARIOS — HARD-CODED
# This is the standard CME SPAN scenario grid.
# price_frac × PSR gives the actual ₹ price move per scenario.
# =============================================================================

SCENARIOS: List[Scenario] = [
    # ── Normal scenarios (weight = 1.0) ──────────────────────────────────
    # Price UP scenarios
    Scenario( 1, "Price +1/3 PSR, Vol UP",   +1/3, "UP",   1.0),
    Scenario( 2, "Price +1/3 PSR, Vol DOWN", +1/3, "DOWN", 1.0),
    Scenario( 3, "Price +2/3 PSR, Vol UP",   +2/3, "UP",   1.0),
    Scenario( 4, "Price +2/3 PSR, Vol DOWN", +2/3, "DOWN", 1.0),
    Scenario( 5, "Price +3/3 PSR, Vol UP",   +1.0, "UP",   1.0),
    Scenario( 6, "Price +3/3 PSR, Vol DOWN", +1.0, "DOWN", 1.0),

    # Price DOWN scenarios
    Scenario( 7, "Price -1/3 PSR, Vol UP",   -1/3, "UP",   1.0),
    Scenario( 8, "Price -1/3 PSR, Vol DOWN", -1/3, "DOWN", 1.0),
    Scenario( 9, "Price -2/3 PSR, Vol UP",   -2/3, "UP",   1.0),
    Scenario(10, "Price -2/3 PSR, Vol DOWN", -2/3, "DOWN", 1.0),
    Scenario(11, "Price -3/3 PSR, Vol UP",   -1.0, "UP",   1.0),
    Scenario(12, "Price -3/3 PSR, Vol DOWN", -1.0, "DOWN", 1.0),

    # No price move scenarios (vol only — matters for options)
    Scenario(13, "Price   0,    Vol UP",       0.0, "UP",   1.0),
    Scenario(14, "Price   0,    Vol DOWN",     0.0, "DOWN", 1.0),

    # ── Extreme scenarios (weight = 0.35) ─────────────────────────────────
    # Price moves 2× the scan range — catastrophic but unlikely
    # SPAN only counts 35% of the loss from these scenarios
    Scenario(15, "Price +2×PSR, Vol UP  [EXTREME]", +2.0, "UP",   0.35),
    Scenario(16, "Price -2×PSR, Vol DOWN [EXTREME]", -2.0, "DOWN", 0.35),
]


# =============================================================================
# 2.3 COMPUTE SCENARIO PARAMETERS FOR EACH COMMODITY
# For a given commodity, calculate the actual ₹ price move and vol shift
# under each scenario.
# =============================================================================

def get_scenario_params(
    commodity:   str,
    base_price:  float,
) -> List[dict]:
    """
    Returns a list of 16 dicts, one per scenario, containing:
      - scenario_id, label, weight
      - price_move_rs : actual ₹ move in futures price
      - vol_shift     : actual shift in implied vol (as decimal)
      - hypo_price    : hypothetical futures price under this scenario
    """
    params      = SPAN_PARAMS[commodity]
    psr_rs      = base_price * params["price_scan_pct"]   # ₹ scan range
    vol_up      = params["vol_scan_up"]
    vol_down    = params["vol_scan_down"]

    result = []
    for s in SCENARIOS:
        price_move  = s.price_frac * psr_rs
        vol_shift   = vol_up if s.vol_move == "UP" else -vol_down
        result.append({
            "scenario_id"  : s.scenario_id,
            "label"        : s.label,
            "price_frac"   : s.price_frac,
            "vol_move"     : s.vol_move,
            "weight"       : s.weight,
            "psr_rs"       : round(psr_rs, 2),
            "price_move_rs": round(price_move, 2),
            "hypo_price"   : round(base_price + price_move, 2),
            "vol_shift"    : round(vol_shift, 4),
        })
    return result


# =============================================================================
# 2.4 PRINT SCENARIO GRID FOR A GIVEN COMMODITY
# =============================================================================

def print_scenario_grid(commodity: str, base_price: float):
    params = get_scenario_params(commodity, base_price)
    psr    = SPAN_PARAMS[commodity]["price_scan_pct"] * 100

    print(f"\n{'='*72}")
    print(f"  SCENARIO GRID — {commodity}")
    print(f"  Base Price : ₹{base_price:,.2f}  |  "
          f"PSR : {psr:.0f}%  |  "
          f"PSR (₹) : ₹{base_price * SPAN_PARAMS[commodity]['price_scan_pct']:,.2f}")
    print(f"{'='*72}")
    print(f"  {'S#':<4} {'Label':<35} {'PriceFrac':>10} {'VolMove':>8} "
          f"{'PriceMv(₹)':>12} {'HypoPrice':>12} {'VolShift':>9} {'Wt':>5}")
    print(f"  {'-'*95}")

    for p in params:
        extreme = " ★" if p["weight"] < 1.0 else "  "
        print(f"  S{p['scenario_id']:<3} {p['label']:<35} "
              f"{p['price_frac']:>+10.4f} {p['vol_move']:>8} "
              f"₹{p['price_move_rs']:>10,.2f} "
              f"₹{p['hypo_price']:>10,.2f} "
              f"{p['vol_shift']:>+9.4f} "
              f"{p['weight']:>4.2f}{extreme}")

    print(f"\n  ★ = Extreme scenario (35% weight — catastrophic but unlikely)")
    print(f"{'='*72}")


# =============================================================================
# 2.5 SUMMARY TABLE — ALL COMMODITIES IN PORTFOLIO
# Shows the PSR (₹) for every commodity — this is the
# maximum single-day price move SPAN will test
# =============================================================================

def print_all_scenario_parameters(portfolio: Portfolio):
    print(f"\n{'='*72}")
    print(f"  SPAN SCENARIO PARAMETERS — ALL COMMODITIES")
    print(f"{'='*72}")
    print(f"\n  {'Commodity':<24} {'Base Price':>12} {'PSR%':>6} "
          f"{'PSR(₹)':>10} {'VolUp':>7} {'VolDn':>7} "
          f"{'Max UP(₹)':>12} {'Max DN(₹)':>12}")
    print(f"  {'-'*93}")

    seen = set()
    for inst in portfolio.all_instruments:
        if inst.inst_type != "Future":
            continue
        key = (inst.commodity, inst.contract_month)
        if key in seen:
            continue
        seen.add(key)

        p        = SPAN_PARAMS[inst.commodity]
        price    = inst.futures_price
        psr_rs   = price * p["price_scan_pct"]
        max_up   = price + 2 * psr_rs    # extreme scenario
        max_dn   = price - 2 * psr_rs    # extreme scenario

        print(f"  {inst.commodity:<24} "
              f"₹{price:>10,.2f} "
              f"{p['price_scan_pct']*100:>5.0f}% "
              f"₹{psr_rs:>9,.2f} "
              f"{p['vol_scan_up']*100:>6.0f}% "
              f"{p['vol_scan_down']*100:>6.0f}% "
              f"₹{max_up:>10,.2f} "
              f"₹{max_dn:>10,.2f}")

    print(f"\n  NOTE: Max UP/DN = Extreme scenario (2× PSR, weight 35%)")
    print(f"  Normal scenarios test moves up to ±1× PSR (weight 100%)")
    print(f"{'='*72}")


# =============================================================================
# 2.6 RUN
# =============================================================================

if __name__ == "__main__":

    # Load portfolio from Step 1
    portfolio = load_portfolio(PORTFOLIO_PATH)

    # Print scenario parameters for all commodities
    print_all_scenario_parameters(portfolio)

    # Print full 16-scenario grid for two representative commodities
    # One spice (high price, 6% PSR) and one cotton (lower price, 5% PSR)
    print_scenario_grid("Jeera",          22285.0)
    print_scenario_grid("Kapas (Cotton)",  1682.0)


  SPAN SCENARIO PARAMETERS — ALL COMMODITIES

  Commodity                  Base Price   PSR%     PSR(₹)   VolUp   VolDn    Max UP(₹)    Max DN(₹)
  ---------------------------------------------------------------------------------------------
  Turmeric                 ₹ 16,220.00     6% ₹   973.20      4%      4% ₹ 18,166.40 ₹ 14,273.60
  Coriander                ₹ 13,350.00     6% ₹   801.00      4%      4% ₹ 14,952.00 ₹ 11,748.00
  Jeera                    ₹ 22,285.00     6% ₹ 1,337.10      4%      4% ₹ 24,959.20 ₹ 19,610.80
  Turmeric                 ₹ 16,450.00     6% ₹   987.00      4%      4% ₹ 18,424.00 ₹ 14,476.00
  Guar Gum                 ₹ 10,756.00     6% ₹   645.36      4%      4% ₹ 12,046.72 ₹  9,465.28
  Guar Seed                ₹  5,715.00     6% ₹   342.90      4%      4% ₹  6,400.80 ₹  5,029.20
  Guar Seed                ₹  5,800.00     6% ₹   348.00      4%      4% ₹  6,496.00 ₹  5,104.00
  Castor Seed              ₹  6,392.00     6% ₹   383.52      4%      4% ₹  7,

In [4]:
# =============================================================================
# MCX SPAN DUMMY MODEL
# STEP 3: PRICE SCAN RISK (PSR)
# Reprice every instrument under all 16 scenarios
# Compute scenario P&L → find worst-case loss = Scan Risk
# =============================================================================

# Paste Steps 1 and 2 above this before running

# =============================================================================
# 3.1 REPRICING LOGIC
#
# FUTURES repricing (simple — purely price driven):
#   P&L = signed_qty × lot_size_qt × price_move_rs
#   signed_qty: +ve = long (profits when price rises)
#               -ve = short (profits when price falls)
#
# OPTIONS repricing (simplified — no Black-Scholes):
#   We use the DELTA-VEGA approximation:
#   P&L = (delta_pnl + vega_pnl) × signed_qty × lot_size_qt
#
#   Delta P&L = delta × price_move_rs
#     → How much option value changes due to price move
#     → delta is from portfolio (e.g. 0.52 for Jeera ATM call)
#
#   Vega P&L  = vega_approx × vol_shift
#     → How much option value changes due to vol move
#     → vega_approx = option_premium × 0.30
#        (simplified: vega ≈ 30% of premium sensitivity to vol)
#
#   For SHORT options: signed_qty is negative
#     → short call loses when price rises (delta effect)
#     → short option loses when vol rises (vega effect)
#
# Weighted P&L = raw_pnl × scenario_weight
#   → Extreme scenarios (S15/S16) only count at 35%
# =============================================================================

def reprice_instrument(
    inst:          Instrument,
    price_move_rs: float,
    vol_shift:     float,
    weight:        float,
) -> dict:
    """
    Reprices a single instrument under one scenario.
    Returns raw P&L, weighted P&L, and component breakdown.
    """
    if inst.inst_type == "Future":
        # Futures: purely price-driven
        raw_pnl    = inst.signed_qty * inst.lot_size_qt * price_move_rs
        delta_pnl  = raw_pnl
        vega_pnl   = 0.0

    elif inst.inst_type == "Option":
        # Options: delta effect + vega effect
        delta      = inst.delta or 0.0
        premium    = inst.option_premium or 0.0

        # Delta P&L: how option value moves with price
        delta_pnl  = delta * price_move_rs

        # Vega P&L: how option value moves with volatility
        # vega_approx = 30% of premium as sensitivity proxy
        vega_approx = premium * 0.30
        vega_pnl    = vega_approx * vol_shift * 100
        # × 100 converts vol_shift (decimal) to vol points scale

        # Total option P&L per unit per quintal
        unit_pnl   = delta_pnl + vega_pnl

        # Scale by position size
        raw_pnl    = inst.signed_qty * inst.lot_size_qt * unit_pnl
    else:
        raw_pnl   = 0.0
        delta_pnl = 0.0
        vega_pnl  = 0.0

    weighted_pnl = raw_pnl * weight

    return {
        "row_id"      : inst.row_id,
        "commodity"   : inst.commodity,
        "inst_type"   : inst.inst_type,
        "direction"   : inst.direction,
        "signed_qty"  : inst.signed_qty,
        "delta_pnl"   : round(delta_pnl, 2),
        "vega_pnl"    : round(vega_pnl, 2),
        "raw_pnl"     : round(raw_pnl, 2),
        "weight"      : weight,
        "weighted_pnl": round(weighted_pnl, 2),
    }


# =============================================================================
# 3.2 SCENARIO P&L FOR ONE COMBINED COMMODITY
# Runs all 16 scenarios for all instruments in a CC group
# Returns a 16-row table of total portfolio P&L per scenario
# =============================================================================

def compute_cc_scenario_pnl(cc: CombinedCommodity) -> list:
    """
    For each of the 16 scenarios:
      1. Get price move + vol shift for each instrument's commodity
      2. Reprice every instrument
      3. Sum P&L across all instruments → CC-level scenario P&L
    Returns list of 16 dicts (one per scenario).
    """
    scenario_results = []

    for s in SCENARIOS:
        instruments_pnl = []
        total_raw       = 0.0
        total_weighted  = 0.0

        for inst in cc.instruments:
            # Get commodity-specific scenario parameters
            commodity = inst.commodity
            if commodity not in SPAN_PARAMS:
                continue

            params = SPAN_PARAMS[commodity]

            # Base price: futures use futures_price,
            # options use strike_price as proxy for underlying
            if inst.inst_type == "Future":
                base_price = inst.futures_price
            else:
                # For options, use strike as base (simplified)
                base_price = inst.strike_price or inst.futures_price or 0

            psr_rs     = base_price * params["price_scan_pct"]
            price_move = s.price_frac * psr_rs
            vol_shift  = (params["vol_scan_up"]
                          if s.vol_move == "UP"
                          else -params["vol_scan_down"])

            result = reprice_instrument(inst, price_move, vol_shift, s.weight)
            instruments_pnl.append(result)
            total_raw      += result["raw_pnl"]
            total_weighted += result["weighted_pnl"]

        scenario_results.append({
            "scenario_id"    : s.scenario_id,
            "label"          : s.label,
            "price_frac"     : s.price_frac,
            "vol_move"       : s.vol_move,
            "weight"         : s.weight,
            "total_raw_pnl"  : round(total_raw, 2),
            "total_wtd_pnl"  : round(total_weighted, 2),
            "instruments"    : instruments_pnl,
        })

    return scenario_results


# =============================================================================
# 3.3 SCAN RISK = WORST WEIGHTED LOSS ACROSS ALL 16 SCENARIOS
# This is THE core SPAN number — the starting point for margin
# =============================================================================

def compute_scan_risk(scenario_results: list) -> dict:
    """
    Scan Risk = max(0, -min(weighted_pnl across all 16 scenarios))
    i.e. the largest LOSS (most negative P&L) across all scenarios.
    Floored at 0 — scan risk cannot be negative.
    """
    worst = min(scenario_results, key=lambda x: x["total_wtd_pnl"])
    best  = max(scenario_results, key=lambda x: x["total_wtd_pnl"])

    scan_risk = max(0.0, -worst["total_wtd_pnl"])

    return {
        "scan_risk"          : round(scan_risk, 2),
        "worst_scenario_id"  : worst["scenario_id"],
        "worst_scenario_lbl" : worst["label"],
        "worst_pnl"          : worst["total_wtd_pnl"],
        "best_scenario_id"   : best["scenario_id"],
        "best_pnl"           : best["total_wtd_pnl"],
    }


# =============================================================================
# 3.4 RUN PSR FOR ENTIRE PORTFOLIO
# =============================================================================

def compute_portfolio_psr(portfolio: Portfolio) -> dict:
    """
    Computes scan risk for every CC group.
    Returns dict: cc_name → {scenario_results, scan_risk_result}
    """
    psr_results = {}
    for cc_name, cc in portfolio.combined_commodities.items():
        scenario_results = compute_cc_scenario_pnl(cc)
        scan_risk_result = compute_scan_risk(scenario_results)
        psr_results[cc_name] = {
            "cc"              : cc,
            "scenarios"       : scenario_results,
            "scan_risk"       : scan_risk_result,
        }
    return psr_results


# =============================================================================
# 3.5 PRINT OUTPUTS
# =============================================================================

def print_scenario_pnl_table(cc_name: str, scenario_results: list):
    """Prints the 16-scenario P&L table for one CC group."""
    print(f"\n{'='*72}")
    print(f"  SCENARIO P&L TABLE — {cc_name}")
    print(f"{'='*72}")
    print(f"  {'S#':<4} {'Label':<35} {'RawP&L(₹)':>12} "
          f"{'Weight':>7} {'WtdP&L(₹)':>12}")
    print(f"  {'-'*72}")

    for s in scenario_results:
        extreme = " ★" if s["weight"] < 1.0 else "  "
        print(f"  S{s['scenario_id']:<3} {s['label']:<35} "
              f"₹{s['total_raw_pnl']:>10,.2f} "
              f"{s['weight']:>7.2f} "
              f"₹{s['total_wtd_pnl']:>10,.2f}{extreme}")

    print(f"  {'─'*72}")
    worst = min(scenario_results, key=lambda x: x["total_wtd_pnl"])
    best  = max(scenario_results, key=lambda x: x["total_wtd_pnl"])
    print(f"  WORST: S{worst['scenario_id']} → "
          f"₹{worst['total_wtd_pnl']:>10,.2f}  ← SCAN RISK BASIS")
    print(f"  BEST : S{best['scenario_id']}  → "
          f"₹{best['total_wtd_pnl']:>10,.2f}")
    print(f"{'='*72}")


def print_instrument_breakdown(cc_name: str, scenario_results: list,
                                worst_sid: int):
    """Prints per-instrument P&L for the worst scenario."""
    worst = next(s for s in scenario_results
                 if s["scenario_id"] == worst_sid)

    print(f"\n  INSTRUMENT BREAKDOWN — {cc_name} — "
          f"Worst Scenario S{worst_sid}")
    print(f"  ({worst['label']})")
    print(f"  {'─'*75}")
    print(f"  {'Row':<4} {'Commodity':<24} {'Type':<7} {'Dir':<5} "
          f"{'Qty':>4} {'DeltaPnL':>12} {'VegaPnL':>10} {'RawPnL':>12}")
    print(f"  {'-'*75}")

    for inst_r in worst["instruments"]:
        print(f"  {inst_r['row_id']:<4} {inst_r['commodity']:<24} "
              f"{inst_r['inst_type']:<7} {inst_r['direction']:<5} "
              f"{abs(inst_r['signed_qty']):>4} "
              f"₹{inst_r['delta_pnl']:>10,.2f} "
              f"₹{inst_r['vega_pnl']:>8,.2f} "
              f"₹{inst_r['raw_pnl']:>10,.2f}")

    print(f"  {'─'*75}")
    total = sum(i["raw_pnl"] for i in worst["instruments"])
    print(f"  {'TOTAL':<43} ₹{total:>10,.2f}")


def print_psr_summary(psr_results: dict):
    """Prints the final PSR summary across all CC groups."""
    print(f"\n{'='*72}")
    print(f"  PRICE SCAN RISK (PSR) SUMMARY — ALL CC GROUPS")
    print(f"{'='*72}")
    print(f"\n  {'CC Group':<20} {'Scan Risk (₹)':>15} "
          f"{'Worst Scen':>12} {'Worst P&L':>14}")
    print(f"  {'-'*63}")

    total_psr = 0.0
    for cc_name, res in psr_results.items():
        sr = res["scan_risk"]
        total_psr += sr["scan_risk"]
        print(f"  {cc_name:<20} ₹{sr['scan_risk']:>13,.2f} "
              f"  S{sr['worst_scenario_id']:<10} "
              f"₹{sr['worst_pnl']:>12,.2f}")

    print(f"  {'─'*63}")
    print(f"  {'TOTAL GROSS PSR':<20} ₹{total_psr:>13,.2f}")
    print(f"\n  NOTE: This is GROSS scan risk before any credits.")
    print(f"  Steps 4–7 will apply ICSC, SOM, Inter-CC and NOV adjustments.")
    print(f"{'='*72}")


# =============================================================================
# 3.6 RUN
# =============================================================================

if __name__ == "__main__":

    # Load portfolio
    portfolio = load_portfolio(PORTFOLIO_PATH)

    # Compute PSR for all CC groups
    psr_results = compute_portfolio_psr(portfolio)

    # Print scenario P&L table + instrument breakdown for each CC group
    for cc_name, res in psr_results.items():
        print_scenario_pnl_table(cc_name, res["scenarios"])
        print_instrument_breakdown(
            cc_name,
            res["scenarios"],
            res["scan_risk"]["worst_scenario_id"]
        )

    # Final PSR summary
    print_psr_summary(psr_results)


  SCENARIO P&L TABLE — Spices CC
  S#   Label                                  RawP&L(₹)  Weight    WtdP&L(₹)
  ------------------------------------------------------------------------
  S1   Price +1/3 PSR, Vol UP              ₹ 58,408.00    1.00 ₹ 58,408.00  
  S2   Price +1/3 PSR, Vol DOWN            ₹119,968.00    1.00 ₹119,968.00  
  S3   Price +2/3 PSR, Vol UP              ₹147,596.00    1.00 ₹147,596.00  
  S4   Price +2/3 PSR, Vol DOWN            ₹209,156.00    1.00 ₹209,156.00  
  S5   Price +3/3 PSR, Vol UP              ₹236,784.00    1.00 ₹236,784.00  
  S6   Price +3/3 PSR, Vol DOWN            ₹298,344.00    1.00 ₹298,344.00  
  S7   Price -1/3 PSR, Vol UP              ₹-119,968.00    1.00 ₹-119,968.00  
  S8   Price -1/3 PSR, Vol DOWN            ₹-58,408.00    1.00 ₹-58,408.00  
  S9   Price -2/3 PSR, Vol UP              ₹-209,156.00    1.00 ₹-209,156.00  
  S10  Price -2/3 PSR, Vol DOWN            ₹-147,596.00    1.00 ₹-147,596.00  
  S11  Price -3/3 PSR, Vol UP         

In [5]:
# =============================================================================
# MCX SPAN DUMMY MODEL
# STEP 4: INTRA-COMMODITY SPREAD CHARGE (ICSC)
# Detects calendar spread pairs within the same commodity
# Applies a fixed charge to account for residual basis risk
# =============================================================================

# Paste Steps 1, 2, and 3 above this before running

# =============================================================================
# 4.1 WHAT IS THE INTRA-COMMODITY SPREAD CHARGE?
#
# When you hold opposite positions in the SAME commodity but DIFFERENT
# expiry months — e.g. Buy Turmeric Jun-26 AND Sell Turmeric Apr-26 —
# SPAN gives you a CREDIT because the two legs partially offset.
#
# BUT — there is still RESIDUAL RISK:
#   The two months may NOT move by the exact same amount.
#   This difference is called "BASIS RISK" or "ROLL RISK".
#   Example: Jun-26 Turmeric might fall ₹500 while Apr-26 falls only ₹480.
#   The spread position still has a ₹20 residual exposure.
#
# The ICSC is a FIXED CHARGE added back to cover this residual basis risk.
# It is NOT a penalty — it is a fair charge for the risk that remains
# even after the offset is recognized.
#
# In our portfolio we have TWO calendar spreads:
#   1. Turmeric  : Buy Jun-26 (row 9)  vs  Sell Apr-26 (row 10)
#   2. Guar Seed : Buy Jun-26 (row 11) vs  Sell Apr-26 (row 12)
#
# ICSC formula:
#   num_spread_pairs = min(long_lots, short_lots) in same commodity
#   ICSC = num_spread_pairs × spread_charge (₹ per pair)
#
# Both legs must be:
#   - Same commodity (e.g. Turmeric)
#   - Same CC group (e.g. Spices CC)
#   - DIFFERENT contract months
#   - One Buy + one Sell
# =============================================================================

# =============================================================================
# 4.2 CALENDAR SPREAD DETECTOR
# Scans each CC group for calendar spread pairs
# =============================================================================

def detect_calendar_spreads(cc: CombinedCommodity) -> list:
    """
    Detects calendar spread pairs within a CC group.
    A calendar spread = same commodity, different months,
    one long + one short.

    Returns a list of spread pair dicts.
    """
    spreads_found = []

    # Group futures by commodity
    by_commodity: Dict[str, List[Instrument]] = {}
    for inst in cc.futures:
        if inst.commodity not in by_commodity:
            by_commodity[inst.commodity] = []
        by_commodity[inst.commodity].append(inst)

    for commodity, insts in by_commodity.items():
        # Need at least 2 contracts in different months
        months = set(i.contract_month for i in insts)
        if len(months) < 2:
            continue

        # Separate longs and shorts
        longs  = [i for i in insts if i.direction == "Buy"]
        shorts = [i for i in insts if i.direction == "Sell"]

        if not longs or not shorts:
            continue

        # For each long-short pair with different months — it's a spread
        for long_leg in longs:
            for short_leg in shorts:
                if long_leg.contract_month == short_leg.contract_month:
                    continue  # same month = not a calendar spread

                # Number of spread pairs = min(long lots, short lots)
                num_pairs = min(long_leg.qty_lots, short_leg.qty_lots)

                if num_pairs == 0:
                    continue

                params        = SPAN_PARAMS.get(commodity, {})
                charge_per_pair = params.get("spread_charge", 5000)
                total_charge  = num_pairs * charge_per_pair

                spreads_found.append({
                    "commodity"       : commodity,
                    "cc_group"        : cc.name,
                    "long_row"        : long_leg.row_id,
                    "long_month"      : long_leg.contract_month,
                    "long_lots"       : long_leg.qty_lots,
                    "short_row"       : short_leg.row_id,
                    "short_month"     : short_leg.contract_month,
                    "short_lots"      : short_leg.qty_lots,
                    "num_pairs"       : num_pairs,
                    "charge_per_pair" : charge_per_pair,
                    "total_icsc"      : total_charge,
                })

    return spreads_found


# =============================================================================
# 4.3 COMPUTE ICSC FOR ENTIRE PORTFOLIO
# =============================================================================

def compute_portfolio_icsc(portfolio: Portfolio) -> dict:
    """
    Detects all calendar spreads across all CC groups.
    Returns per-CC and portfolio-level ICSC totals.
    """
    icsc_results = {}
    total_icsc   = 0.0

    for cc_name, cc in portfolio.combined_commodities.items():
        spreads = detect_calendar_spreads(cc)
        cc_icsc = sum(s["total_icsc"] for s in spreads)
        total_icsc += cc_icsc

        icsc_results[cc_name] = {
            "spreads"  : spreads,
            "cc_icsc"  : cc_icsc,
        }

    icsc_results["__total__"] = total_icsc
    return icsc_results


# =============================================================================
# 4.4 ADJUSTED SCAN RISK AFTER ICSC
# SPAN logic: Scan Risk is REDUCED by the calendar spread offset credit
# but the ICSC charge is ADDED BACK to cover basis risk
#
# Adjusted Scan Risk = Gross Scan Risk - Spread Credit + ICSC
#
# For our simplified model:
#   Spread Credit ≈ scan risk of matched spread lots
#   Since we don't compute per-lot scan risk separately,
#   we apply the ICSC as a NET addition to the gross PSR.
#   (In real SPAN, the credit and charge are computed simultaneously —
#    net effect is that calendar spreads always pay some charge.)
#
# Net effect on margin:
#   Calendar spreads REDUCE gross margin (credit for offset)
#   but ADD the ICSC charge (for residual basis risk)
#   Net ICSC impact = ICSC charge (the credit is implicit in
#   how futures legs offset each other in the scenario P&L)
# =============================================================================

def compute_adjusted_scan_risk(
    psr_results:  dict,
    icsc_results: dict,
) -> dict:
    """
    Adds ICSC to the gross scan risk per CC group.
    Returns adjusted scan risk per CC group.
    """
    adjusted = {}

    for cc_name, psr in psr_results.items():
        gross_sr = psr["scan_risk"]["scan_risk"]
        cc_icsc  = icsc_results.get(cc_name, {}).get("cc_icsc", 0.0)
        adj_sr   = gross_sr + cc_icsc

        adjusted[cc_name] = {
            "gross_scan_risk" : gross_sr,
            "icsc_charge"     : cc_icsc,
            "adjusted_sr"     : round(adj_sr, 2),
        }

    return adjusted


# =============================================================================
# 4.5 PRINT OUTPUTS
# =============================================================================

def print_icsc_detail(icsc_results: dict):
    print(f"\n{'='*72}")
    print(f"  INTRA-COMMODITY SPREAD CHARGE (ICSC) — DETAIL")
    print(f"{'='*72}")

    any_found = False
    for cc_name, res in icsc_results.items():
        if cc_name == "__total__":
            continue
        if not res["spreads"]:
            print(f"\n  {cc_name}: No calendar spreads detected.")
            continue

        any_found = True
        print(f"\n  CC GROUP: {cc_name}")
        print(f"  {'─'*65}")

        for sp in res["spreads"]:
            print(f"\n  Commodity       : {sp['commodity']}")
            print(f"  Long  leg       : Row {sp['long_row']}  "
                  f"{sp['long_month']}  ×{sp['long_lots']} lots")
            print(f"  Short leg       : Row {sp['short_row']}  "
                  f"{sp['short_month']}  ×{sp['short_lots']} lots")
            print(f"  Spread pairs    : min({sp['long_lots']}, "
                  f"{sp['short_lots']}) = {sp['num_pairs']}")
            print(f"  Charge per pair : ₹{sp['charge_per_pair']:,.0f}")
            print(f"  ICSC Total      : ₹{sp['total_icsc']:,.0f}  "
                  f"({sp['num_pairs']} × ₹{sp['charge_per_pair']:,.0f})")

        print(f"\n  CC ICSC Total   : ₹{res['cc_icsc']:,.0f}")

    total = icsc_results.get("__total__", 0)
    print(f"\n{'─'*72}")
    print(f"  PORTFOLIO TOTAL ICSC : ₹{total:,.0f}")
    print(f"{'='*72}")


def print_adjusted_scan_risk(adjusted: dict, total_icsc: float):
    print(f"\n{'='*72}")
    print(f"  SCAN RISK AFTER ICSC ADJUSTMENT")
    print(f"{'='*72}")
    print(f"\n  {'CC Group':<20} {'Gross SR (₹)':>14} "
          f"{'ICSC (₹)':>12} {'Adjusted SR (₹)':>16}")
    print(f"  {'-'*64}")

    total_gross = 0.0
    total_adj   = 0.0

    for cc_name, res in adjusted.items():
        total_gross += res["gross_scan_risk"]
        total_adj   += res["adjusted_sr"]
        print(f"  {cc_name:<20} "
              f"₹{res['gross_scan_risk']:>12,.2f} "
              f"₹{res['icsc_charge']:>10,.2f} "
              f"₹{res['adjusted_sr']:>14,.2f}")

    print(f"  {'─'*64}")
    print(f"  {'TOTAL':<20} "
          f"₹{total_gross:>12,.2f} "
          f"₹{total_icsc:>10,.2f} "
          f"₹{total_adj:>14,.2f}")

    print(f"\n  WHAT ICSC MEANS IN PRACTICE:")
    print(f"  {'─'*60}")
    print(f"  Gross PSR (before ICSC) : ₹{total_gross:>10,.2f}")
    print(f"  (+) ICSC charge         : ₹{total_icsc:>10,.2f}")
    print(f"  ─────────────────────────────────────────")
    print(f"  Adjusted Scan Risk      : ₹{total_adj:>10,.2f}")
    print(f"\n  NOTE: Calendar spreads REDUCE gross risk (legs offset each")
    print(f"  other in scenario P&L — already reflected in Step 3 output).")
    print(f"  ICSC is added back as a FLAT CHARGE for residual basis risk.")
    print(f"  Net effect: spread traders pay a small 'basis risk' premium.")
    print(f"{'='*72}")


# =============================================================================
# 4.6 RUN
# =============================================================================

if __name__ == "__main__":

    # Load portfolio
    portfolio = load_portfolio(PORTFOLIO_PATH)

    # Step 3 — PSR (needed for adjusted scan risk)
    psr_results = compute_portfolio_psr(portfolio)

    # Step 4 — ICSC
    icsc_results = compute_portfolio_icsc(portfolio)
    total_icsc   = icsc_results["__total__"]

    # Adjusted scan risk = gross PSR + ICSC
    adjusted_sr  = compute_adjusted_scan_risk(psr_results, icsc_results)

    # Print
    print_icsc_detail(icsc_results)
    print_adjusted_scan_risk(adjusted_sr, total_icsc)


  INTRA-COMMODITY SPREAD CHARGE (ICSC) — DETAIL

  CC GROUP: Spices CC
  ─────────────────────────────────────────────────────────────────

  Commodity       : Turmeric
  Long  leg       : Row 9  Jun-26  ×1 lots
  Short leg       : Row 10  Apr-26  ×1 lots
  Spread pairs    : min(1, 1) = 1
  Charge per pair : ₹5,000
  ICSC Total      : ₹5,000  (1 × ₹5,000)

  CC ICSC Total   : ₹5,000

  CC GROUP: Guar CC
  ─────────────────────────────────────────────────────────────────

  Commodity       : Guar Seed
  Long  leg       : Row 11  Jun-26  ×2 lots
  Short leg       : Row 12  Apr-26  ×2 lots
  Spread pairs    : min(2, 2) = 2
  Charge per pair : ₹4,000
  ICSC Total      : ₹8,000  (2 × ₹4,000)

  CC ICSC Total   : ₹8,000

  Castor CC: No calendar spreads detected.

  Cotton CC: No calendar spreads detected.

────────────────────────────────────────────────────────────────────────
  PORTFOLIO TOTAL ICSC : ₹13,000

  SCAN RISK AFTER ICSC ADJUSTMENT

  CC Group               Gross SR (₹)     IC

In [6]:
# =============================================================================
# MCX SPAN DUMMY MODEL
# STEP 5: SHORT OPTION MINIMUM (SOM)
# Floor margin for all short option positions
# Ensures short options always carry minimum margin
# even when scenario scan risk is near zero
# =============================================================================

# Paste Steps 1, 2, 3, and 4 above this before running

# =============================================================================
# 5.1 WHAT IS THE SHORT OPTION MINIMUM?
#
# Problem this solves:
#   Deep OTM short options have very low delta (e.g. −0.04 for row 21).
#   Under all 16 SPAN scenarios, the price never moves enough to make
#   the option significantly ITM — so the scenario scan risk is TINY.
#
#   BUT — the option seller has collected premium and has UNLIMITED
#   downside if the market moves sharply (a "black swan" event).
#   The 16 scenarios don't capture this tail risk fully.
#
# Solution — SOM:
#   SPAN says: "Even if your scenario scan risk is ₹0,
#   you MUST post a minimum margin as a short option seller."
#
#   SOM = som_rate × contract_value
#   contract_value = strike_price × lot_size_qt × qty_lots
#   som_rate = 1.5% for all commodities in our model
#
# Final margin for each short option:
#   option_margin = max(scan_risk_of_option, SOM)
#
# This is the "SOM is binding" situation — when SOM > scan_risk,
# the exchange takes the higher number. This is exactly what
# happens for rows 21 and 22 (deep OTM shorts).
#
# For near-ATM short options (rows 14, 16, 18, 20):
#   scan_risk > SOM → SOM is NOT binding → use scan_risk
#
# For deep OTM short options (rows 21, 22):
#   scan_risk ≈ 0 → SOM IS binding → use SOM
# =============================================================================

# =============================================================================
# 5.2 EXTRACT OPTION SCAN RISK PER INSTRUMENT
# We need the scan risk contribution of each SHORT option individually
# (not the CC-level total) to compare against SOM
# =============================================================================

def compute_option_individual_scan_risk(
    inst:         Instrument,
    scenarios:    list,
) -> float:
    """
    Computes the standalone scan risk for a single option position.
    Uses the pre-computed CC scenario results — extracts this
    instrument's contribution from each scenario's instrument list,
    then finds the worst weighted loss.
    """
    worst_wtd_pnl = 0.0

    for scen in scenarios:
        # Find this instrument's P&L in this scenario
        for inst_r in scen["instruments"]:
            if inst_r["row_id"] == inst.row_id:
                wtd_pnl = inst_r["weighted_pnl"]
                if wtd_pnl < worst_wtd_pnl:
                    worst_wtd_pnl = wtd_pnl
                break

    scan_risk = max(0.0, -worst_wtd_pnl)
    return round(scan_risk, 2)


# =============================================================================
# 5.3 COMPUTE SOM FOR A SINGLE SHORT OPTION
# =============================================================================

def compute_som(inst: Instrument) -> dict:
    """
    Computes the Short Option Minimum for one short option position.
    Returns SOM in ₹.

    SOM = som_rate × strike_price × lot_size_qt × qty_lots
    """
    if not inst.is_short or inst.inst_type != "Option":
        return {
            "row_id"       : inst.row_id,
            "commodity"    : inst.commodity,
            "option_type"  : inst.option_type,
            "strike"       : inst.strike_price,
            "qty_lots"     : inst.qty_lots,
            "lot_size_qt"  : inst.lot_size_qt,
            "som_rate"     : 0.0,
            "som"          : 0.0,
            "applies"      : False,
        }

    params   = SPAN_PARAMS.get(inst.commodity, {})
    som_rate = params.get("som_rate", 0.015)

    # Contract value based on strike price
    # (SOM uses strike as the reference price — the option's intrinsic anchor)
    contract_val = inst.strike_price * inst.lot_size_qt * inst.qty_lots
    som          = som_rate * contract_val

    return {
        "row_id"       : inst.row_id,
        "commodity"    : inst.commodity,
        "option_type"  : inst.option_type,
        "direction"    : inst.direction,
        "strike"       : inst.strike_price,
        "qty_lots"     : inst.qty_lots,
        "lot_size_qt"  : inst.lot_size_qt,
        "delta"        : inst.delta,
        "premium"      : inst.option_premium,
        "som_rate"     : som_rate,
        "contract_val" : round(contract_val, 2),
        "som"          : round(som, 2),
        "applies"      : True,
        "span_role"    : inst.span_role,
    }


# =============================================================================
# 5.4 COMPUTE SOM FOR ENTIRE PORTFOLIO
# For each short option: compute SOM + individual scan risk
# Final option margin = max(scan_risk, SOM)
# =============================================================================

def compute_portfolio_som(
    portfolio:   Portfolio,
    psr_results: dict,
) -> dict:
    """
    Computes SOM for all short options across the portfolio.
    Returns per-CC and portfolio-level SOM results.
    """
    som_results  = {}
    total_som_binding    = 0.0   # sum of SOMs where SOM > scan_risk
    total_scan_binding   = 0.0   # sum of scan risks where scan_risk > SOM

    for cc_name, cc in portfolio.combined_commodities.items():
        cc_scenarios = psr_results[cc_name]["scenarios"]
        cc_som_rows  = []

        for inst in cc.short_options:
            # Individual scan risk for this option
            ind_scan_risk = compute_option_individual_scan_risk(
                inst, cc_scenarios
            )

            # SOM for this option
            som_result = compute_som(inst)
            som_val    = som_result["som"]

            # Final option margin = max(scan_risk, SOM)
            final_margin = max(ind_scan_risk, som_val)
            som_binding  = som_val > ind_scan_risk

            cc_som_rows.append({
                **som_result,
                "ind_scan_risk" : ind_scan_risk,
                "final_margin"  : round(final_margin, 2),
                "som_binding"   : som_binding,
                "binding_label" : "SOM BINDING ★" if som_binding
                                  else "Scan Risk Binding",
            })

            if som_binding:
                total_som_binding  += final_margin
            else:
                total_scan_binding += final_margin

        cc_total_final = sum(r["final_margin"] for r in cc_som_rows)
        cc_som_added   = sum(
            max(0, r["som"] - r["ind_scan_risk"])
            for r in cc_som_rows
            if r["som_binding"]
        )

        som_results[cc_name] = {
            "rows"          : cc_som_rows,
            "cc_total_final": round(cc_total_final, 2),
            "cc_som_added"  : round(cc_som_added, 2),
        }

    som_results["__totals__"] = {
        "som_binding_total"  : round(total_som_binding, 2),
        "scan_binding_total" : round(total_scan_binding, 2),
        "grand_total"        : round(total_som_binding + total_scan_binding, 2),
    }

    return som_results


# =============================================================================
# 5.5 PRINT OUTPUTS
# =============================================================================

def print_som_detail(som_results: dict):
    print(f"\n{'='*78}")
    print(f"  SHORT OPTION MINIMUM (SOM) — FULL DETAIL")
    print(f"{'='*78}")
    print(f"\n  SOM Formula: SOM = som_rate × strike × lot_size_qt × qty_lots")
    print(f"  SOM Rate   : 1.5% for all commodities in this model")
    print(f"  Final Margin per option = max(Scan Risk, SOM)")

    for cc_name, res in som_results.items():
        if cc_name == "__totals__":
            continue
        if not res["rows"]:
            print(f"\n  {cc_name}: No short options.")
            continue

        print(f"\n  {'─'*75}")
        print(f"  CC GROUP: {cc_name}")
        print(f"  {'─'*75}")
        print(f"  {'Row':<4} {'Commodity':<22} {'Type':<5} "
              f"{'Strike':>8} {'Qty':>4} {'LotQt':>6} "
              f"{'ContVal':>12} {'SOM(₹)':>10} "
              f"{'ScanRisk':>10} {'Final(₹)':>10} {'Status'}")
        print(f"  {'-'*105}")

        for r in res["rows"]:
            print(f"  {r['row_id']:<4} {r['commodity']:<22} "
                  f"{r['option_type']:<5} "
                  f"₹{r['strike']:>7,.0f} "
                  f"{r['qty_lots']:>4} "
                  f"{r['lot_size_qt']:>6.0f} "
                  f"₹{r['contract_val']:>10,.0f} "
                  f"₹{r['som']:>8,.0f} "
                  f"₹{r['ind_scan_risk']:>8,.0f} "
                  f"₹{r['final_margin']:>8,.0f} "
                  f"  {r['binding_label']}")

        print(f"\n  CC Total Short Option Margin : "
              f"₹{res['cc_total_final']:>10,.2f}")

    # Portfolio summary
    totals = som_results["__totals__"]
    print(f"\n{'='*78}")
    print(f"  SOM PORTFOLIO SUMMARY")
    print(f"{'='*78}")
    print(f"\n  Short options where SOM is binding  : "
          f"₹{totals['som_binding_total']:>10,.2f}")
    print(f"  Short options where Scan Risk binds : "
          f"₹{totals['scan_binding_total']:>10,.2f}")
    print(f"  {'─'*48}")
    print(f"  Total Short Option Margin           : "
          f"₹{totals['grand_total']:>10,.2f}")
    print(f"{'='*78}")


def print_som_vs_scan_comparison(som_results: dict):
    """Shows clearly which options triggered SOM and why."""
    print(f"\n{'='*78}")
    print(f"  SOM vs SCAN RISK — BINDING ANALYSIS")
    print(f"{'='*78}")
    print(f"\n  {'Row':<4} {'Commodity':<22} {'Delta':>7} "
          f"{'ScanRisk':>10} {'SOM':>10} {'Final':>10} {'Winner':>18}")
    print(f"  {'-'*78}")

    for cc_name, res in som_results.items():
        if cc_name == "__totals__":
            continue
        for r in res["rows"]:
            winner = "★ SOM WINS" if r["som_binding"] else "  Scan Risk wins"
            print(f"  {r['row_id']:<4} {r['commodity']:<22} "
                  f"{r['delta']:>7.2f} "
                  f"₹{r['ind_scan_risk']:>8,.0f} "
                  f"₹{r['som']:>8,.0f} "
                  f"₹{r['final_margin']:>8,.0f} "
                  f"  {winner}")

    print(f"\n  KEY INSIGHT:")
    print(f"  Deep OTM options (delta ≈ 0) have near-zero scan risk")
    print(f"  because the 16 scenarios don't move the price enough")
    print(f"  to bring them ITM. SOM floors the margin to protect")
    print(f"  against black swan events beyond the scenario grid.")
    print(f"{'='*78}")


# =============================================================================
# 5.6 RUN
# =============================================================================

if __name__ == "__main__":

    # Load portfolio
    portfolio = load_portfolio(PORTFOLIO_PATH)

    # Step 3 — PSR
    psr_results = compute_portfolio_psr(portfolio)

    # Step 4 — ICSC
    icsc_results = compute_portfolio_icsc(portfolio)
    adjusted_sr  = compute_adjusted_scan_risk(psr_results, icsc_results)

    # Step 5 — SOM
    som_results  = compute_portfolio_som(portfolio, psr_results)

    # Print
    print_som_detail(som_results)
    print_som_vs_scan_comparison(som_results)


  SHORT OPTION MINIMUM (SOM) — FULL DETAIL

  SOM Formula: SOM = som_rate × strike × lot_size_qt × qty_lots
  SOM Rate   : 1.5% for all commodities in this model
  Final Margin per option = max(Scan Risk, SOM)

  ───────────────────────────────────────────────────────────────────────────
  CC GROUP: Spices CC
  ───────────────────────────────────────────────────────────────────────────
  Row  Commodity              Type    Strike  Qty  LotQt      ContVal     SOM(₹)   ScanRisk   Final(₹) Status
  ---------------------------------------------------------------------------------------------------------
  14   Jeera                  Call  ₹ 23,000    2     30 ₹ 1,380,000 ₹  20,700 ₹  96,264 ₹  96,264   Scan Risk Binding
  21   Jeera                  Call  ₹ 27,000    3     30 ₹ 2,430,000 ₹  36,450 ₹  15,012 ₹  36,450   SOM BINDING ★

  CC Total Short Option Margin : ₹132,714.00

  ───────────────────────────────────────────────────────────────────────────
  CC GROUP: Guar CC
  ───────────

In [7]:
# =============================================================================
# MCX SPAN DUMMY MODEL
# STEP 6: INTER-COMMODITY SPREAD CREDIT (ICCS)
# Margin reduction for correlated commodity pairs within a CC group
# =============================================================================

# Paste Steps 1, 2, 3, 4, and 5 above this before running

# =============================================================================
# 6.1 WHAT IS THE INTER-COMMODITY SPREAD CREDIT?
#
# Within a Combined Commodity (CC) group, some commodities are
# economically correlated. For example:
#   Guar Gum and Guar Seed — Guar Gum IS made from Guar Seed.
#   If Guar Seed price rises, Guar Gum price also tends to rise.
#
#   Cotton Seed Oil Cake and Kapas (Cotton) — both are cotton-related.
#   They tend to move together.
#
# If you hold LONG Guar Gum AND LONG Guar Seed — both go up/down together.
# That is NOT diversification — that is concentration. No credit given.
#
# But if the NET DELTA of the two commodities OPPOSES each other
# (e.g. one is net long and one has options that create a net short delta),
# SPAN gives a PARTIAL CREDIT because the positions partially offset.
#
# HOW THE CREDIT WORKS:
#   Step 1: Compute net dollar delta for each commodity in the pair
#   Step 2: Check if deltas are in OPPOSITE directions
#           (one positive, one negative = opposing)
#   Step 3: Credit = credit_rate × min(|delta_leg1|, |delta_leg2|)
#           We take the SMALLER side — you can only offset up to
#           the size of the smaller exposure
#   Step 4: Subtract credit from adjusted scan risk of that CC group
#
# In our model:
#   Guar CC  : Guar Gum (net long futures + net long put option)
#              vs Guar Seed (net long futures + short put option)
#   Cotton CC: Cotton Seed Oil Cake (net long futures)
#              vs Kapas (net long futures + mixed options)
#
# Credit rate = 30% in our model
# (MCX uses exchange-defined credit tables — we simplify to flat 30%)
#
# IMPORTANT: This credit REDUCES margin.
# It is the ONLY step in our model that gives margin RELIEF.
# =============================================================================

# =============================================================================
# 6.2 COMPUTE NET DOLLAR DELTA PER COMMODITY WITHIN A CC GROUP
# Net delta captures the directional exposure of ALL instruments
# (futures + options) for a single commodity
# =============================================================================

def compute_commodity_net_delta(
    commodity:  str,
    cc:         CombinedCommodity,
) -> dict:
    """
    Computes net dollar delta for a single commodity within a CC group.

    For futures:
      delta_contribution = signed_qty × lot_size_qt × futures_price

    For options:
      delta_contribution = signed_qty × lot_size_qt × delta × strike_price
      (delta from portfolio — already signed correctly:
       long call: +delta, short call: -delta
       long put:  -delta, short put:  +delta)

    Net delta > 0 = net long exposure (loses if price falls)
    Net delta < 0 = net short exposure (loses if price rises)
    """
    instruments = [i for i in cc.instruments if i.commodity == commodity]

    net_delta     = 0.0
    fut_delta     = 0.0
    opt_delta     = 0.0
    breakdown     = []

    for inst in instruments:
        if inst.inst_type == "Future":
            contribution = (
                inst.signed_qty
                * inst.lot_size_qt
                * inst.futures_price
            )
            fut_delta += contribution

        elif inst.inst_type == "Option":
            # delta is already signed in portfolio
            # (negative for short calls, negative for long puts etc.)
            ref_price    = inst.strike_price or inst.futures_price or 0
            contribution = (
                inst.signed_qty
                * inst.lot_size_qt
                * (inst.delta or 0)
                * ref_price
            )
            opt_delta += contribution

        breakdown.append({
            "row_id"      : inst.row_id,
            "inst_type"   : inst.inst_type,
            "direction"   : inst.direction,
            "signed_qty"  : inst.signed_qty,
            "delta"       : inst.delta,
            "contribution": round(contribution, 2),
        })

        net_delta += contribution

    return {
        "commodity"  : commodity,
        "net_delta"  : round(net_delta, 2),
        "fut_delta"  : round(fut_delta, 2),
        "opt_delta"  : round(opt_delta, 2),
        "direction"  : "LONG" if net_delta > 0 else "SHORT",
        "breakdown"  : breakdown,
    }


# =============================================================================
# 6.3 COMPUTE INTER-CC SPREAD CREDIT FOR ONE PAIR
# =============================================================================

def compute_inter_cc_credit(
    cc:          CombinedCommodity,
    pair_def:    dict,
    adjusted_sr: float,
) -> dict:
    """
    Computes the inter-commodity spread credit for one correlated pair.

    pair_def contains: leg1, leg2, credit_rate
    adjusted_sr: the CC group's scan risk after ICSC (from Step 4)
    """
    leg1_name   = pair_def["leg1"]
    leg2_name   = pair_def["leg2"]
    credit_rate = pair_def["credit_rate"]

    # Net delta for each leg
    delta1 = compute_commodity_net_delta(leg1_name, cc)
    delta2 = compute_commodity_net_delta(leg2_name, cc)

    d1 = delta1["net_delta"]
    d2 = delta2["net_delta"]

    # Check if deltas OPPOSE each other
    # Opposing = one positive, one negative
    opposing = (d1 > 0 and d2 < 0) or (d1 < 0 and d2 > 0)

    if opposing:
        # Credit = credit_rate × min(|delta1|, |delta2|)
        smaller_delta = min(abs(d1), abs(d2))
        credit        = credit_rate * smaller_delta
        credit_note   = (f"Opposing deltas — credit applies\n"
                         f"    min(|{d1:,.0f}|, |{d2:,.0f}|) = "
                         f"{smaller_delta:,.0f} × {credit_rate*100:.0f}%")
    else:
        credit        = 0.0
        credit_note   = "Same direction deltas — no credit"

    # Adjusted SR after credit — floor at 0
    adj_sr_after_credit = max(0.0, adjusted_sr - credit)

    return {
        "cc_group"           : cc.name,
        "leg1"               : leg1_name,
        "leg2"               : leg2_name,
        "delta1"             : delta1,
        "delta2"             : delta2,
        "d1"                 : d1,
        "d2"                 : d2,
        "opposing"           : opposing,
        "credit_rate"        : credit_rate,
        "credit"             : round(credit, 2),
        "credit_note"        : credit_note,
        "adjusted_sr_before" : round(adjusted_sr, 2),
        "adjusted_sr_after"  : round(adj_sr_after_credit, 2),
    }


# =============================================================================
# 6.4 COMPUTE INTER-CC CREDIT FOR ENTIRE PORTFOLIO
# =============================================================================

def compute_portfolio_inter_cc(
    portfolio:   Portfolio,
    adjusted_sr: dict,
) -> dict:
    """
    Computes inter-CC spread credits for all defined pairs.
    Returns per-CC credit results and updated adjusted scan risks.
    """
    inter_results  = {}
    total_credit   = 0.0

    # Updated adjusted SR — starts from Step 4 values
    updated_adj_sr = {
        cc_name: res["adjusted_sr"]
        for cc_name, res in adjusted_sr.items()
    }

    for pair_def in INTER_CC_SPREADS:
        cc_name = pair_def["cc_group"]

        if cc_name not in portfolio.combined_commodities:
            continue

        cc     = portfolio.combined_commodities[cc_name]
        adj_sr = updated_adj_sr.get(cc_name, 0.0)

        result = compute_inter_cc_credit(cc, pair_def, adj_sr)

        inter_results[cc_name] = result
        total_credit          += result["credit"]

        # Update the adjusted SR for this CC group
        updated_adj_sr[cc_name] = result["adjusted_sr_after"]

    # CC groups with no inter-CC pair get no credit
    for cc_name in portfolio.combined_commodities:
        if cc_name not in inter_results:
            inter_results[cc_name] = {
                "cc_group"           : cc_name,
                "credit"             : 0.0,
                "adjusted_sr_before" : adjusted_sr.get(cc_name, {})
                                       .get("adjusted_sr", 0),
                "adjusted_sr_after"  : adjusted_sr.get(cc_name, {})
                                       .get("adjusted_sr", 0),
                "opposing"           : None,
                "no_pair_defined"    : True,
            }

    inter_results["__total_credit__"] = round(total_credit, 2)
    inter_results["__updated_adj_sr__"] = updated_adj_sr

    return inter_results


# =============================================================================
# 6.5 PRINT OUTPUTS
# =============================================================================

def print_inter_cc_detail(inter_results: dict):
    print(f"\n{'='*75}")
    print(f"  INTER-COMMODITY SPREAD CREDIT (ICCS) — FULL DETAIL")
    print(f"{'='*75}")
    print(f"\n  Credit Formula: ICCS = credit_rate × min(|ΔLeg1|, |ΔLeg2|)")
    print(f"  Credit Rate   : 30% for all pairs in this model")
    print(f"  Condition     : Deltas must OPPOSE each other to qualify")

    for cc_name, res in inter_results.items():
        if cc_name.startswith("__"):
            continue

        print(f"\n  {'─'*70}")
        print(f"  CC GROUP: {cc_name}")
        print(f"  {'─'*70}")

        if res.get("no_pair_defined"):
            print(f"  No inter-commodity pair defined for this CC group.")
            print(f"  Adjusted SR unchanged: ₹{res['adjusted_sr_after']:,.2f}")
            continue

        # Delta breakdown
        print(f"\n  LEG 1: {res['leg1']}")
        d1 = res["delta1"]
        print(f"    Futures delta : ₹{d1['fut_delta']:>12,.2f}")
        print(f"    Options delta : ₹{d1['opt_delta']:>12,.2f}")
        print(f"    NET DELTA     : ₹{d1['net_delta']:>12,.2f}  "
              f"({d1['direction']})")

        print(f"\n  LEG 2: {res['leg2']}")
        d2 = res["delta2"]
        print(f"    Futures delta : ₹{d2['fut_delta']:>12,.2f}")
        print(f"    Options delta : ₹{d2['opt_delta']:>12,.2f}")
        print(f"    NET DELTA     : ₹{d2['net_delta']:>12,.2f}  "
              f"({d2['direction']})")

        print(f"\n  CREDIT ASSESSMENT:")
        print(f"    Deltas opposing : {'YES — credit applies' if res['opposing'] else 'NO — same direction, no credit'}")

        if res["opposing"]:
            smaller = min(abs(res["d1"]), abs(res["d2"]))
            print(f"    Smaller delta   : ₹{smaller:>12,.2f}")
            print(f"    Credit rate     : {res['credit_rate']*100:.0f}%")
            print(f"    ICCS Credit     : ₹{res['credit']:>12,.2f}")
        else:
            print(f"    ICCS Credit     : ₹0.00")

        print(f"\n  ADJUSTED SR:")
        print(f"    Before ICCS     : ₹{res['adjusted_sr_before']:>12,.2f}")
        print(f"    (-) ICCS Credit : ₹{res['credit']:>12,.2f}")
        print(f"    After ICCS      : ₹{res['adjusted_sr_after']:>12,.2f}")

    total = inter_results["__total_credit__"]
    print(f"\n{'─'*75}")
    print(f"  TOTAL INTER-CC CREDIT : ₹{total:>10,.2f}")
    print(f"{'='*75}")


def print_inter_cc_summary(
    inter_results: dict,
    adjusted_sr:   dict,
    icsc_results:  dict,
    psr_results:   dict,
):
    """Running margin tally after Steps 3, 4, 5, 6."""
    updated = inter_results["__updated_adj_sr__"]
    total_credit = inter_results["__total_credit__"]

    print(f"\n{'='*75}")
    print(f"  MARGIN BUILD-UP AFTER STEP 6 — CC GROUP SUMMARY")
    print(f"{'='*75}")
    print(f"\n  {'CC Group':<20} {'Gross PSR':>12} {'ICSC':>8} "
          f"{'AdjSR':>12} {'ICCS Credit':>13} {'Final SR':>12}")
    print(f"  {'-'*79}")

    total_gross = 0.0
    total_icsc  = 0.0
    total_adj   = 0.0
    total_final = 0.0

    for cc_name in psr_results:
        gross  = psr_results[cc_name]["scan_risk"]["scan_risk"]
        icsc   = icsc_results.get(cc_name, {}).get("cc_icsc", 0.0)
        adj    = adjusted_sr.get(cc_name, {}).get("adjusted_sr", gross)
        credit = inter_results.get(cc_name, {}).get("credit", 0.0)
        final  = updated.get(cc_name, adj)

        total_gross += gross
        total_icsc  += icsc
        total_adj   += adj
        total_final += final

        print(f"  {cc_name:<20} "
              f"₹{gross:>10,.0f} "
              f"₹{icsc:>6,.0f} "
              f"₹{adj:>10,.0f} "
              f"₹{credit:>11,.0f} "
              f"₹{final:>10,.0f}")

    print(f"  {'─'*79}")
    print(f"  {'TOTAL':<20} "
          f"₹{total_gross:>10,.0f} "
          f"₹{total_icsc:>6,.0f} "
          f"₹{total_adj:>10,.0f} "
          f"₹{total_credit:>11,.0f} "
          f"₹{total_final:>10,.0f}")

    print(f"\n  RUNNING MARGIN WATERFALL:")
    print(f"  {'─'*55}")
    print(f"  Gross PSR  (Step 3)          : ₹{total_gross:>10,.2f}")
    print(f"  (+) ICSC   (Step 4)          : ₹{total_icsc:>10,.2f}")
    print(f"  (-) ICCS   (Step 6)          : ₹{total_credit:>10,.2f}")
    print(f"  ─────────────────────────────────────────────")
    print(f"  Adjusted Scan Risk           : ₹{total_final:>10,.2f}")
    print(f"\n  NOTE: Step 7 (NOV) will make the final margin adjustment.")
    print(f"{'='*75}")


# =============================================================================
# 6.6 RUN
# =============================================================================

if __name__ == "__main__":

    # Load portfolio
    portfolio = load_portfolio(PORTFOLIO_PATH)

    # Step 3 — PSR
    psr_results  = compute_portfolio_psr(portfolio)

    # Step 4 — ICSC
    icsc_results = compute_portfolio_icsc(portfolio)
    adjusted_sr  = compute_adjusted_scan_risk(psr_results, icsc_results)

    # Step 5 — SOM (computed but not needed for SR adjustment)
    som_results  = compute_portfolio_som(portfolio, psr_results)

    # Step 6 — Inter-CC Credit
    inter_results = compute_portfolio_inter_cc(portfolio, adjusted_sr)

    # Print
    print_inter_cc_detail(inter_results)
    print_inter_cc_summary(
        inter_results, adjusted_sr, icsc_results, psr_results
    )


  INTER-COMMODITY SPREAD CREDIT (ICCS) — FULL DETAIL

  Credit Formula: ICCS = credit_rate × min(|ΔLeg1|, |ΔLeg2|)
  Credit Rate   : 30% for all pairs in this model
  Condition     : Deltas must OPPOSE each other to qualify

  ──────────────────────────────────────────────────────────────────────
  CC GROUP: Guar CC
  ──────────────────────────────────────────────────────────────────────

  LEG 1: Guar Gum
    Futures delta : ₹1,075,600.00
    Options delta : ₹ -504,000.00
    NET DELTA     : ₹  571,600.00  (LONG)

  LEG 2: Guar Seed
    Futures delta : ₹1,151,500.00
    Options delta : ₹   13,500.00
    NET DELTA     : ₹1,165,000.00  (LONG)

  CREDIT ASSESSMENT:
    Deltas opposing : NO — same direction, no credit
    ICCS Credit     : ₹0.00

  ADJUSTED SR:
    Before ICCS     : ₹  112,196.00
    (-) ICCS Credit : ₹        0.00
    After ICCS      : ₹  112,196.00

  ──────────────────────────────────────────────────────────────────────
  CC GROUP: Cotton CC
  ────────────────────────

In [8]:
# =============================================================================
# MCX SPAN DUMMY MODEL
# STEP 7: NET OPTION VALUE (NOV) + FINAL MARGIN ASSEMBLY
# The last adjustment — then the complete final margin report
# =============================================================================

# Paste Steps 1, 2, 3, 4, 5, and 6 above this before running

# =============================================================================
# 7.1 WHAT IS NET OPTION VALUE (NOV)?
#
# Every option has a market value (its premium × lot size × quantity).
# SPAN recognises that:
#
#   LONG options  = you PAID premium upfront → you have an ASSET
#                   This asset reduces your margin requirement
#                   because if you close the position, you get money back
#
#   SHORT options = you RECEIVED premium upfront → you have a LIABILITY
#                   This liability INCREASES your margin requirement
#                   because you may have to pay out more than you received
#
# NOV = Net Option Value
#     = Total value of LONG options − Total value of SHORT options
#
# If NOV > 0 (long options dominate):
#   → You have more option assets than liabilities
#   → This REDUCES your final margin
#
# If NOV < 0 (short options dominate):
#   → You owe more than you hold
#   → This INCREASES your final margin
#
# ANOV = Adjusted NOV
#   Real SPAN caps the benefit from long options.
#   You cannot reduce margin by MORE than the scan risk.
#   ANOV = max(NOV, -scan_risk)   [cannot give more credit than risk]
#   Then: Final Margin = Adjusted Scan Risk + Short Option Margin - ANOV
#                        (if ANOV > 0, it reduces margin)
#                        (if ANOV < 0, it increases margin)
#
# =============================================================================

# =============================================================================
# 7.2 COMPUTE NOV PER CC GROUP
# =============================================================================

def compute_nov(cc: CombinedCommodity) -> dict:
    """
    Computes Net Option Value for a CC group.

    LOV = sum(premium × lot_size_qt × qty_lots) for all LONG options
    SOV = sum(premium × lot_size_qt × qty_lots) for all SHORT options
    NOV = LOV - SOV

    If NOV > 0: long options exceed short → margin credit
    If NOV < 0: short options exceed long → margin debit
    """
    lov_rows = []
    sov_rows = []
    lov = 0.0
    sov = 0.0

    for inst in cc.options:
        opt_value = (
            inst.option_premium
            * inst.lot_size_qt
            * inst.qty_lots
        )

        if inst.is_long:
            lov += opt_value
            lov_rows.append({
                "row_id"    : inst.row_id,
                "commodity" : inst.commodity,
                "opt_type"  : inst.option_type,
                "direction" : inst.direction,
                "qty"       : inst.qty_lots,
                "premium"   : inst.option_premium,
                "lot_qt"    : inst.lot_size_qt,
                "value"     : round(opt_value, 2),
            })
        else:
            sov += opt_value
            sov_rows.append({
                "row_id"    : inst.row_id,
                "commodity" : inst.commodity,
                "opt_type"  : inst.option_type,
                "direction" : inst.direction,
                "qty"       : inst.qty_lots,
                "premium"   : inst.option_premium,
                "lot_qt"    : inst.lot_size_qt,
                "value"     : round(opt_value, 2),
            })

    nov = lov - sov

    return {
        "cc_group" : cc.name,
        "lov"      : round(lov, 2),
        "sov"      : round(sov, 2),
        "nov"      : round(nov, 2),
        "lov_rows" : lov_rows,
        "sov_rows" : sov_rows,
        "nov_sign" : "Credit (reduces margin)" if nov > 0
                     else "Debit (increases margin)" if nov < 0
                     else "Zero",
    }


# =============================================================================
# 7.3 COMPUTE ANOV (ADJUSTED NOV)
# ANOV caps the benefit so long options cannot reduce margin
# below zero for any CC group
# =============================================================================

def compute_anov(nov: float, scan_risk: float) -> dict:
    """
    ANOV = NOV, but capped so it cannot exceed the scan risk.
    (You cannot get MORE credit than the scan risk of the CC group.)

    If NOV > 0 (credit):
        ANOV = min(NOV, scan_risk)   ← cap the benefit
    If NOV <= 0 (debit):
        ANOV = NOV                   ← full debit applies

    Margin impact:
        final_margin = scan_risk + short_option_margin - ANOV
    """
    if nov > 0:
        anov      = min(nov, scan_risk)
        capped    = nov > scan_risk
        cap_note  = f"Capped at scan risk ₹{scan_risk:,.0f}" if capped else "No cap needed"
    else:
        anov      = nov
        capped    = False
        cap_note  = "Debit — no cap applied"

    return {
        "nov"      : round(nov, 2),
        "anov"     : round(anov, 2),
        "capped"   : capped,
        "cap_note" : cap_note,
    }


# =============================================================================
# 7.4 FINAL MARGIN ASSEMBLY
# Brings together ALL steps into one final number per CC group
# and portfolio total
# =============================================================================

def compute_final_margin(
    portfolio:     Portfolio,
    psr_results:   dict,
    icsc_results:  dict,
    adjusted_sr:   dict,
    inter_results: dict,
    som_results:   dict,
) -> dict:
    """
    Final SPAN margin per CC group:

    = Adjusted Scan Risk (post ICSC, post ICCS)
    + Short Option Margin (max of scan risk or SOM per short option)
    - ANOV (adjusted net option value)

    Portfolio total = sum across all CC groups
    """
    updated_adj_sr = inter_results["__updated_adj_sr__"]
    final_results  = {}
    portfolio_total = 0.0

    for cc_name, cc in portfolio.combined_commodities.items():

        # Adjusted scan risk after Steps 4 + 6
        adj_sr = updated_adj_sr.get(cc_name, 0.0)

        # Short option margin from Step 5
        cc_som_data     = som_results.get(cc_name, {})
        short_opt_margin = cc_som_data.get("cc_total_final", 0.0)

        # NOV for this CC group
        nov_result = compute_nov(cc)
        nov        = nov_result["nov"]

        # ANOV — capped version of NOV
        anov_result = compute_anov(nov, adj_sr)
        anov        = anov_result["anov"]

        # Final margin = adj_sr + short_opt_margin - ANOV
        final_margin = adj_sr + short_opt_margin - anov
        final_margin = max(0.0, final_margin)   # floor at zero

        portfolio_total += final_margin

        final_results[cc_name] = {
            "adj_scan_risk"   : round(adj_sr, 2),
            "short_opt_margin": round(short_opt_margin, 2),
            "nov_result"      : nov_result,
            "anov_result"     : anov_result,
            "nov"             : round(nov, 2),
            "anov"            : round(anov, 2),
            "final_margin"    : round(final_margin, 2),
        }

    final_results["__portfolio_total__"] = round(portfolio_total, 2)
    return final_results


# =============================================================================
# 7.5 PRINT OUTPUTS
# =============================================================================

def print_nov_detail(final_results: dict):
    print(f"\n{'='*75}")
    print(f"  NET OPTION VALUE (NOV) — DETAIL PER CC GROUP")
    print(f"{'='*75}")
    print(f"\n  NOV Formula : NOV = LOV − SOV")
    print(f"  LOV = sum(premium × lot_qt × qty) for LONG options")
    print(f"  SOV = sum(premium × lot_qt × qty) for SHORT options")

    for cc_name, res in final_results.items():
        if cc_name.startswith("__"):
            continue

        nov_r = res["nov_result"]
        print(f"\n  {'─'*65}")
        print(f"  CC GROUP: {cc_name}")
        print(f"  {'─'*65}")

        print(f"\n  LONG OPTIONS (LOV):")
        if nov_r["lov_rows"]:
            for r in nov_r["lov_rows"]:
                print(f"    Row {r['row_id']}: {r['commodity']} "
                      f"{r['opt_type']} "
                      f"×{r['qty']} lots  "
                      f"₹{r['premium']} × {r['lot_qt']:.0f}qt × "
                      f"{r['qty']} = ₹{r['value']:>10,.2f}")
        else:
            print(f"    None")
        print(f"    LOV Total : ₹{nov_r['lov']:>10,.2f}")

        print(f"\n  SHORT OPTIONS (SOV):")
        if nov_r["sov_rows"]:
            for r in nov_r["sov_rows"]:
                print(f"    Row {r['row_id']}: {r['commodity']} "
                      f"{r['opt_type']} "
                      f"×{r['qty']} lots  "
                      f"₹{r['premium']} × {r['lot_qt']:.0f}qt × "
                      f"{r['qty']} = ₹{r['value']:>10,.2f}")
        else:
            print(f"    None")
        print(f"    SOV Total : ₹{nov_r['sov']:>10,.2f}")

        anov_r = res["anov_result"]
        print(f"\n  NOV  = LOV − SOV = "
              f"₹{nov_r['lov']:,.2f} − ₹{nov_r['sov']:,.2f} "
              f"= ₹{nov_r['nov']:,.2f}  ({nov_r['nov_sign']})")
        print(f"  ANOV = ₹{anov_r['anov']:,.2f}  [{anov_r['cap_note']}]")

    print(f"{'='*75}")


def print_final_margin_report(
    final_results:  dict,
    psr_results:    dict,
    icsc_results:   dict,
    inter_results:  dict,
):
    print(f"\n{'='*75}")
    print(f"  FINAL SPAN MARGIN — PER CC GROUP")
    print(f"{'='*75}")
    print(f"\n  Formula: Final Margin = Adj Scan Risk + Short Opt Margin − ANOV")
    print(f"\n  {'CC Group':<20} {'AdjSR':>10} {'ShortOpt':>10} "
          f"{'NOV':>10} {'ANOV':>10} {'FINAL':>12}")
    print(f"  {'-'*74}")

    total_adj  = 0.0
    total_som  = 0.0
    total_nov  = 0.0
    total_anov = 0.0
    total_fin  = 0.0

    for cc_name, res in final_results.items():
        if cc_name.startswith("__"):
            continue
        total_adj  += res["adj_scan_risk"]
        total_som  += res["short_opt_margin"]
        total_nov  += res["nov"]
        total_anov += res["anov"]
        total_fin  += res["final_margin"]

        print(f"  {cc_name:<20} "
              f"₹{res['adj_scan_risk']:>8,.0f} "
              f"₹{res['short_opt_margin']:>8,.0f} "
              f"₹{res['nov']:>8,.0f} "
              f"₹{res['anov']:>8,.0f} "
              f"₹{res['final_margin']:>10,.0f}")

    print(f"  {'─'*74}")
    print(f"  {'TOTAL':<20} "
          f"₹{total_adj:>8,.0f} "
          f"₹{total_som:>8,.0f} "
          f"₹{total_nov:>8,.0f} "
          f"₹{total_anov:>8,.0f} "
          f"₹{total_fin:>10,.0f}")

    # ── COMPLETE MARGIN WATERFALL ─────────────────────────────────────────
    total_gross = sum(
        psr_results[cc]["scan_risk"]["scan_risk"]
        for cc in psr_results
    )
    total_icsc  = icsc_results["__total__"]
    total_iccs  = inter_results["__total_credit__"]

    print(f"\n{'='*75}")
    print(f"  COMPLETE MARGIN WATERFALL — PORTFOLIO LEVEL")
    print(f"{'='*75}")
    print(f"\n  STEP 3  Gross Price Scan Risk        : ₹{total_gross:>12,.2f}")
    print(f"  STEP 4  (+) Intra-CC Spread Charge   : ₹{total_icsc:>12,.2f}")
    print(f"  STEP 6  (-) Inter-CC Spread Credit   : ₹{total_iccs:>12,.2f}")
    print(f"          {'─'*42}")
    print(f"          Adjusted Scan Risk            : ₹{total_adj:>12,.2f}")
    print(f"  STEP 5  (+) Short Option Margin       : ₹{total_som:>12,.2f}")
    print(f"  STEP 7  (-) ANOV (Net Option Value)   : ₹{total_anov:>12,.2f}")
    print(f"          {'─'*42}")
    print(f"  ★★★     FINAL SPAN MARGIN             : ₹{total_fin:>12,.2f}  ★★★")
    print(f"{'='*75}")

    # ── PER CC WATERFALL ─────────────────────────────────────────────────
    print(f"\n{'='*75}")
    print(f"  PER CC GROUP WATERFALL")
    print(f"{'='*75}")

    for cc_name, res in final_results.items():
        if cc_name.startswith("__"):
            continue
        gross = psr_results[cc_name]["scan_risk"]["scan_risk"]
        icsc  = icsc_results.get(cc_name, {}).get("cc_icsc", 0.0)
        iccs  = inter_results.get(cc_name, {}).get("credit", 0.0)

        print(f"\n  {cc_name}")
        print(f"  {'─'*45}")
        print(f"  Gross PSR                : ₹{gross:>10,.2f}")
        print(f"  (+) ICSC                 : ₹{icsc:>10,.2f}")
        print(f"  (-) ICCS                 : ₹{iccs:>10,.2f}")
        print(f"  Adjusted Scan Risk       : ₹{res['adj_scan_risk']:>10,.2f}")
        print(f"  (+) Short Option Margin  : ₹{res['short_opt_margin']:>10,.2f}")
        print(f"  (-) ANOV                 : ₹{res['anov']:>10,.2f}")
        print(f"  ─────────────────────────────────────")
        print(f"  FINAL MARGIN             : ₹{res['final_margin']:>10,.2f}")

    print(f"\n{'='*75}")
    print(f"  MODEL COMPLETE — ALL 7 STEPS EXECUTED")
    print(f"{'='*75}")
    print(f"\n  Components activated in this portfolio:")
    print(f"  ✅ Step 3 — Price Scan Risk (PSR)          ₹{total_gross:>10,.2f}")
    print(f"  ✅ Step 4 — Intra-CC Spread Charge (ICSC)  ₹{total_icsc:>10,.2f}")
    print(f"  ✅ Step 5 — Short Option Minimum (SOM)     ₹{total_som:>10,.2f}")
    print(f"  ✅ Step 6 — Inter-CC Spread Credit (ICCS)  ₹{total_iccs:>10,.2f}")
    print(f"  ✅ Step 7 — Net Option Value (ANOV)        ₹{total_anov:>10,.2f}")
    print(f"\n  ★★★ TOTAL FINAL SPAN MARGIN : ₹{total_fin:>10,.2f} ★★★")
    print(f"{'='*75}")


# =============================================================================
# 7.6 RUN — FULL PIPELINE
# =============================================================================

if __name__ == "__main__":

    # Load portfolio
    portfolio = load_portfolio(PORTFOLIO_PATH)

    # Step 3 — PSR
    psr_results   = compute_portfolio_psr(portfolio)

    # Step 4 — ICSC
    icsc_results  = compute_portfolio_icsc(portfolio)
    adjusted_sr   = compute_adjusted_scan_risk(psr_results, icsc_results)

    # Step 5 — SOM
    som_results   = compute_portfolio_som(portfolio, psr_results)

    # Step 6 — Inter-CC Credit
    inter_results = compute_portfolio_inter_cc(portfolio, adjusted_sr)

    # Step 7 — NOV + Final Margin
    final_results = compute_final_margin(
        portfolio, psr_results, icsc_results,
        adjusted_sr, inter_results, som_results
    )

    # Print
    print_nov_detail(final_results)
    print_final_margin_report(
        final_results, psr_results, icsc_results, inter_results
    )


  NET OPTION VALUE (NOV) — DETAIL PER CC GROUP

  NOV Formula : NOV = LOV − SOV
  LOV = sum(premium × lot_qt × qty) for LONG options
  SOV = sum(premium × lot_qt × qty) for SHORT options

  ─────────────────────────────────────────────────────────────────
  CC GROUP: Spices CC
  ─────────────────────────────────────────────────────────────────

  LONG OPTIONS (LOV):
    Row 13: Jeera Call ×1 lots  ₹1200.0 × 30qt × 1 = ₹ 36,000.00
    LOV Total : ₹ 36,000.00

  SHORT OPTIONS (SOV):
    Row 14: Jeera Call ×2 lots  ₹900.0 × 30qt × 2 = ₹ 54,000.00
    Row 21: Jeera Call ×3 lots  ₹85.0 × 30qt × 3 = ₹  7,650.00
    SOV Total : ₹ 61,650.00

  NOV  = LOV − SOV = ₹36,000.00 − ₹61,650.00 = ₹-25,650.00  (Debit (increases margin))
  ANOV = ₹-25,650.00  [Debit — no cap applied]

  ─────────────────────────────────────────────────────────────────
  CC GROUP: Guar CC
  ─────────────────────────────────────────────────────────────────

  LONG OPTIONS (LOV):
    Row 15: Guar Gum Put ×1 lots  ₹600.0 × 